# 02 — Train Hybrid YOLOv8n + EfficientNet-B0

Notebook này chạy hybrid pipeline: YOLOv8n binary detector, EfficientNet-B0 classifier, hybrid evaluation, error analysis, Grad-CAM và inference ảnh mới.


# Common setup notes

Trước khi chạy notebook:

1. Bật GPU trong Kaggle: `Settings → Accelerator → GPU`.
2. Thêm dataset TACO/Roboflow ở mục `Add Input`.
3. Nếu dùng W&B online, thêm Kaggle Secret:
   - Name: `WANDB_API_KEY`
   - Value: API key của tài khoản W&B.
4. Sửa biến `REPO_URL`, `WANDB_ENTITY`, `TACO_ROOT` hoặc `ROBOFLOW_ROOT` cho đúng với nhóm bạn.

Các notebook này dùng nguyên command-line scripts trong GitHub repo đã hoàn thiện, không viết lại training logic trong notebook.


In [ ]:
import os
from pathlib import Path

# ===== EDIT ME =====
REPO_URL = "https://github.com/nnnhuan251-hcmus/Detect-Waste-ADN.git"
REPO_DIR = Path("/kaggle/working/Detect-Waste-ADN")
BRANCH = "main"

# W&B
USE_WANDB = False              # đổi True khi smoke W&B đã pass
WANDB_MODE = "online"          # "online" hoặc "offline"
WANDB_ENTITY = ""              # ví dụ: "waste-detection-team"; để "" nếu dùng default entity

# Kaggle data config
DATA_CONFIG = "configs/data/taco_7class.yaml"

# Runtime flags
RUN_SMOKE = True
RUN_FULL = False               # chỉ đổi True sau khi smoke pass
RUN_EVAL = True
RUN_WANDB_EVAL_UPLOAD = False

os.environ["REPO_URL"] = REPO_URL
os.environ["REPO_DIR"] = str(REPO_DIR)
os.environ["BRANCH"] = BRANCH
os.environ["USE_WANDB"] = "1" if USE_WANDB else "0"
os.environ["WANDB_MODE"] = WANDB_MODE
os.environ["WANDB_ENTITY"] = WANDB_ENTITY
os.environ["DATA_CONFIG"] = DATA_CONFIG
os.environ["RUN_SMOKE"] = "1" if RUN_SMOKE else "0"
os.environ["RUN_FULL"] = "1" if RUN_FULL else "0"
os.environ["RUN_EVAL"] = "1" if RUN_EVAL else "0"
os.environ["RUN_WANDB_EVAL_UPLOAD"] = "1" if RUN_WANDB_EVAL_UPLOAD else "0"

print("REPO_DIR:", REPO_DIR)
print("DATA_CONFIG:", DATA_CONFIG)
print("USE_WANDB:", USE_WANDB, "| WANDB_MODE:", WANDB_MODE, "| WANDB_ENTITY:", WANDB_ENTITY or "<default>")
print("RUN_SMOKE:", RUN_SMOKE, "| RUN_FULL:", RUN_FULL, "| RUN_EVAL:", RUN_EVAL)


In [ ]:
%%bash
set -e

cd /kaggle/working

if [ ! -d "$REPO_DIR/.git" ]; then
  echo "Cloning repo..."
  git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"
else
  echo "Repo exists. Pulling latest changes..."
  cd "$REPO_DIR"
  git fetch origin "$BRANCH"
  git checkout "$BRANCH"
  git pull origin "$BRANCH"
fi

cd "$REPO_DIR"

python -m pip install -q --upgrade pip
pip install -q -r requirements.txt

if [ -f requirements-optional.txt ]; then
  pip install -q -r requirements-optional.txt || true
fi

export PYTHONPATH="$REPO_DIR/src:$PYTHONPATH"
echo "PYTHONPATH=$PYTHONPATH"


In [ ]:
import os

if os.environ.get("USE_WANDB") == "1":
    import wandb
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("WANDB_API_KEY")
        os.environ["WANDB_API_KEY"] = key
        wandb.login()
        print("W&B login passed.")
    except Exception as exc:
        print("W&B login failed:", repr(exc))
        print("Switching to offline mode for safety.")
        os.environ["WANDB_MODE"] = "offline"
else:
    print("W&B disabled for this notebook.")


In [ ]:
%%bash
set -e

cd "$REPO_DIR"
export PYTHONPATH="$REPO_DIR/src:$PYTHONPATH"

echo "=== Compile check ==="
python -m compileall -q src scripts

echo "=== Core import check ==="
python - <<'PY'
import torch
from pathlib import Path

from waste_detection.config.config_loader import ConfigLoader
from waste_detection.data.coco_reader import CocoReader
from waste_detection.data.coco_validator import COCOValidator
from waste_detection.data.crop_dataset import CropClassificationDataset
from waste_detection.models.efficientnet_classifier import EfficientNetB0Classifier
from waste_detection.training.detector_trainer import DetectorTrainer
from waste_detection.training.classifier_trainer import ClassifierTrainer
from waste_detection.inference.hybrid_predictor import HybridPredictor
from waste_detection.evaluation.classification_metrics import ClassificationMetrics
from waste_detection.evaluation.hybrid_metrics import HybridMetrics

print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
print("core imports passed")
PY


In [ ]:
%%bash
set -e

cd "$REPO_DIR"
export PYTHONPATH="$REPO_DIR/src:$PYTHONPATH"

echo "=== Detector fixed-size guard ==="
python - <<'PY'
from pathlib import Path
import re

p = Path("src/waste_detection/training/detector_trainer.py")
text = p.read_text(encoding="utf-8")

checks = {
    "imgsz": bool(re.search(r'["\']imgsz["\']\s*:', text)),
    "rect_false": bool(re.search(r'["\']rect["\']\s*:\s*False', text)),
    "multi_scale_false": bool(re.search(r'["\']multi_scale["\']\s*:\s*False', text)),
}

print(checks)

missing = [name for name, ok in checks.items() if not ok]
if missing:
    raise SystemExit(
        "DetectorTrainer chưa khóa fixed image size đủ an toàn. "
        f"Thiếu: {missing}. "
        "Hãy sửa _build_train_args(): đặt imgsz=int(detector.input_size), rect=False, multi_scale=False. "
        "Nếu không sửa, YOLO/RT-DETR có thể nhận ảnh 640 và 1280 trong cùng batch và crash."
    )

print("Detector fixed-size guard passed.")
PY


In [ ]:
%%bash
set -e

cd "$REPO_DIR"
export PYTHONPATH="$REPO_DIR/src:$PYTHONPATH"

python - <<'PY'
from pathlib import Path
import yaml
import copy

src_dir = Path("configs/experiments")
out_dir = src_dir / "smoke"
out_dir.mkdir(parents=True, exist_ok=True)

def load_yaml(path):
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

def save_yaml(path, data):
    with open(path, "w", encoding="utf-8") as f:
        yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)

def make_smoke(src_name, out_name, epochs=1, batch_size=2, num_workers=2, patience=0):
    cfg = load_yaml(src_dir / src_name)
    cfg["experiment"]["name"] = out_name.replace(".yaml", "")
    cfg["experiment"]["description"] = "SMOKE TEST - " + str(cfg["experiment"].get("description", ""))

    cfg.setdefault("training", {})
    cfg["training"]["epochs"] = epochs
    cfg["training"]["batch_size"] = batch_size
    cfg["training"]["num_workers"] = num_workers
    cfg["training"]["patience"] = patience
    cfg["training"]["early_stopping"] = False
    cfg["training"]["mixed_precision"] = True

    if cfg.get("scheduler", {}).get("name") == "cosine_annealing":
        cfg["training"]["epochs"] = max(2, epochs)
        cfg["scheduler"].setdefault("warmup", {})
        cfg["scheduler"]["warmup"]["warmup_epochs"] = 1

    cfg.setdefault("tracking", {})
    cfg["tracking"]["use_wandb"] = False
    cfg["tracking"]["mode"] = "disabled"
    cfg["tracking"]["log_model"] = False
    cfg["tracking"]["watch_model"] = False
    cfg["tracking"]["log_code"] = False

    save_yaml(out_dir / out_name, cfg)

make_smoke("run1_baseline.yaml", "smoke_run1_baseline.yaml", epochs=1, batch_size=2)
make_smoke("run2_strong_aug.yaml", "smoke_run2_strong_aug.yaml", epochs=1, batch_size=2)
make_smoke("run3_freeze_cosine_warmup.yaml", "smoke_run3_freeze_cosine_warmup.yaml", epochs=2, batch_size=2)
make_smoke("run4_balanced_sampler.yaml", "smoke_run4_balanced_sampler.yaml", epochs=1, batch_size=2)

# RT-DETR smoke: force tiny batch
cfg = load_yaml(src_dir / "run1_baseline.yaml")
cfg["experiment"]["name"] = "smoke_run1_rtdetr"
cfg["experiment"]["description"] = "SMOKE TEST - RT-DETR-L tiny batch."
cfg.setdefault("training", {})
cfg["training"]["epochs"] = 1
cfg["training"]["batch_size"] = 1
cfg["training"]["num_workers"] = 2
cfg["training"]["patience"] = 0
cfg["training"]["early_stopping"] = False
cfg["training"]["mixed_precision"] = True
cfg.setdefault("tracking", {})
cfg["tracking"]["use_wandb"] = False
cfg["tracking"]["mode"] = "disabled"
cfg["tracking"]["log_model"] = False
cfg["tracking"]["watch_model"] = False
cfg["tracking"]["log_code"] = False
save_yaml(out_dir / "smoke_run1_rtdetr.yaml", cfg)

print("Smoke configs:")
for p in sorted(out_dir.glob("*.yaml")):
    print(" -", p)
PY


## 1. Kiểm tra dữ liệu processed đã tồn tại


In [ ]:
%%bash
set -e

cd "$REPO_DIR"

# Tự động tìm và giải nén data từ Notebook 01 nếu đã được Add Input
if [ ! -d "data/processed/yolo_binary_waste" ]; then
  echo "Thử tìm và giải nén dữ liệu từ /kaggle/input/..."
  TAR_FILE=$(find /kaggle/input -name "preprocessed_data_and_reports.tar.gz" | head -n 1)
  if [ -n "$TAR_FILE" ]; then
    echo "Tìm thấy: $TAR_FILE"
    tar -xzf "$TAR_FILE" -C .
    echo "Giải nén thành công!"
  else
    echo "❌ KHÔNG tìm thấy preprocessed_data_and_reports.tar.gz."
    echo "👉 Bạn vui lòng nhìn sang menu phải, bấm 'Add Input' -> 'Your Work' -> Chọn Notebook 01 trước nhé."
    exit 1
  fi
fi

test -f data/processed/yolo_binary_waste/data.yaml || (echo "Missing yolo_binary_waste/data.yaml. Chạy notebook 01 trước." && exit 1)
test -d data/processed/crops_7class/train || (echo "Missing crops_7class/train. Chạy notebook 01 trước." && exit 1)

echo "✅ Data ready."
find data/processed -maxdepth 3 -type f -name "data.yaml" -print


## 2. Smoke train hybrid detector và classifier

Chạy nhanh để bắt lỗi trước khi train thật.


In [ ]:
%%bash
set -e

cd "$REPO_DIR"
export PYTHONPATH="$REPO_DIR/src:$PYTHONPATH"

if [ "$RUN_SMOKE" != "1" ]; then
  echo "RUN_SMOKE=0 → skip smoke training."
  exit 0
fi

echo "=== Smoke hybrid detector D1 ==="
python scripts/train/train_detector.py \
  --data-config "$DATA_CONFIG" \
  --model-config configs/models/hybrid_yolov8n_effb0.yaml \
  --experiment-config configs/experiments/smoke/smoke_run1_baseline.yaml

echo "=== Smoke classifier C1 ==="
python scripts/train/train_classifier.py \
  --data-config "$DATA_CONFIG" \
  --model-config configs/models/hybrid_yolov8n_effb0.yaml \
  --experiment-config configs/experiments/smoke/smoke_run1_baseline.yaml

echo "=== Smoke classifier C2/C3/C4 ==="
for RUN in smoke_run2_strong_aug smoke_run3_freeze_cosine_warmup smoke_run4_balanced_sampler
do
  python scripts/train/train_classifier.py \
    --data-config "$DATA_CONFIG" \
    --model-config configs/models/hybrid_yolov8n_effb0.yaml \
    --experiment-config configs/experiments/smoke/${RUN}.yaml
done


## 3. Train thật — Hybrid detectors D1/D2/D3

Đổi `RUN_FULL=True` ở cell đầu sau khi smoke pass.


In [ ]:
%%bash
set -e

cd "$REPO_DIR"
export PYTHONPATH="$REPO_DIR/src:$PYTHONPATH"


WANDB_FLAGS=""
if [ "$USE_WANDB" = "1" ]; then
  WANDB_FLAGS="--use-wandb --wandb-mode $WANDB_MODE"
  if [ -n "$WANDB_ENTITY" ]; then
    WANDB_FLAGS="$WANDB_FLAGS --wandb-entity $WANDB_ENTITY"
  fi
fi
echo "WANDB_FLAGS=$WANDB_FLAGS"


if [ "$RUN_FULL" != "1" ]; then
  echo "RUN_FULL=0 → skip full detector training."
  exit 0
fi

for RUN in run1_baseline run2_strong_aug run3_freeze_cosine_warmup
do
  echo "=== Train hybrid detector: $RUN ==="
  python scripts/train/train_detector.py \
    --data-config "$DATA_CONFIG" \
    --model-config configs/models/hybrid_yolov8n_effb0.yaml \
    --experiment-config configs/experiments/${RUN}.yaml \
    $WANDB_FLAGS
done


## 4. Train thật — EfficientNet-B0 classifiers C1/C2/C3/C4


In [ ]:
%%bash
set -e

cd "$REPO_DIR"
export PYTHONPATH="$REPO_DIR/src:$PYTHONPATH"


WANDB_FLAGS=""
if [ "$USE_WANDB" = "1" ]; then
  WANDB_FLAGS="--use-wandb --wandb-mode $WANDB_MODE"
  if [ -n "$WANDB_ENTITY" ]; then
    WANDB_FLAGS="$WANDB_FLAGS --wandb-entity $WANDB_ENTITY"
  fi
fi
echo "WANDB_FLAGS=$WANDB_FLAGS"


if [ "$RUN_FULL" != "1" ]; then
  echo "RUN_FULL=0 → skip full classifier training."
  exit 0
fi

for RUN in run1_baseline run2_strong_aug run3_freeze_cosine_warmup run4_balanced_sampler
do
  echo "=== Train classifier: $RUN ==="
  python scripts/train/train_classifier.py \
    --data-config "$DATA_CONFIG" \
    --model-config configs/models/hybrid_yolov8n_effb0.yaml \
    --experiment-config configs/experiments/${RUN}.yaml \
    $WANDB_FLAGS
done


## 5. Evaluate classifiers + error analysis


In [ ]:
%%bash
set -e

cd "$REPO_DIR"
export PYTHONPATH="$REPO_DIR/src:$PYTHONPATH"

if [ "$RUN_EVAL" != "1" ]; then
  echo "RUN_EVAL=0 → skip evaluation."
  exit 0
fi

for RUN in run1_baseline run2_strong_aug run3_freeze_cosine_warmup run4_balanced_sampler
do
  CKPT="outputs/checkpoints/efficientnet_b0/${RUN}/best.pth"
  if [ ! -f "$CKPT" ]; then
    echo "Skip classifier eval $RUN: missing $CKPT"
    continue
  fi

  python scripts/eval/evaluate_classifier.py \
    --data-config "$DATA_CONFIG" \
    --model-config configs/models/hybrid_yolov8n_effb0.yaml \
    --weights "$CKPT" \
    --split test
done


## 6. Evaluate hybrid end-to-end H1/H2/H3/H4

Cell này yêu cầu `evaluate_hybrid.py` đã có `--run-name` và dùng `train_annotations_file/val_annotations_file/test_annotations_file`.


In [ ]:
%%bash
set -e

cd "$REPO_DIR"
export PYTHONPATH="$REPO_DIR/src:$PYTHONPATH"

if [ "$RUN_EVAL" != "1" ]; then
  echo "RUN_EVAL=0 → skip hybrid evaluation."
  exit 0
fi

python - <<'PY'
from pathlib import Path
txt = Path("scripts/eval/evaluate_hybrid.py").read_text(encoding="utf-8")
if "--run-name" not in txt:
    raise SystemExit("evaluate_hybrid.py chưa có --run-name. Sửa trước để tránh overwrite metrics.")
if "train_annotations_path" in txt:
    raise SystemExit("evaluate_hybrid.py còn dùng train_annotations_path. Đổi sang train_annotations_file/val_annotations_file/test_annotations_file.")
print("evaluate_hybrid.py guard passed.")
PY

run_hybrid_eval () {
  NAME="$1"
  DET="$2"
  CLF="$3"

  if [ ! -f "$DET" ]; then
    echo "Skip $NAME: missing detector $DET"
    return 0
  fi
  if [ ! -f "$CLF" ]; then
    echo "Skip $NAME: missing classifier $CLF"
    return 0
  fi

  python scripts/eval/evaluate_hybrid.py \
    --data-config "$DATA_CONFIG" \
    --model-config configs/models/hybrid_yolov8n_effb0.yaml \
    --detector-weights "$DET" \
    --classifier-weights "$CLF" \
    --split test \
    --run-name "$NAME" \
    --save-visualizations \
    --max-visualizations 30
}

run_hybrid_eval "H1_run1_baseline" \
  "outputs/checkpoints/hybrid_yolov8n_effb0/run1_baseline/weights/best.pt" \
  "outputs/checkpoints/efficientnet_b0/run1_baseline/best.pth"

run_hybrid_eval "H2_run2_strong_aug" \
  "outputs/checkpoints/hybrid_yolov8n_effb0/run2_strong_aug/weights/best.pt" \
  "outputs/checkpoints/efficientnet_b0/run2_strong_aug/best.pth"

run_hybrid_eval "H3_run3_freeze_cosine_warmup" \
  "outputs/checkpoints/hybrid_yolov8n_effb0/run3_freeze_cosine_warmup/weights/best.pt" \
  "outputs/checkpoints/efficientnet_b0/run3_freeze_cosine_warmup/best.pth"

run_hybrid_eval "H4_run4_balanced_sampler" \
  "outputs/checkpoints/hybrid_yolov8n_effb0/run2_strong_aug/weights/best.pt" \
  "outputs/checkpoints/efficientnet_b0/run4_balanced_sampler/best.pth"


## 7. Inference demo trên ảnh TACO và ảnh mới


In [ ]:
%%bash
set -e

cd "$REPO_DIR"
export PYTHONPATH="$REPO_DIR/src:$PYTHONPATH"

DET="outputs/checkpoints/hybrid_yolov8n_effb0/run2_strong_aug/weights/best.pt"
CLF="outputs/checkpoints/efficientnet_b0/run4_balanced_sampler/best.pth"

if [ ! -f "$DET" ] || [ ! -f "$CLF" ]; then
  echo "Skip inference demo: missing best H4 weights."
  exit 0
fi

IMG=$(find data/raw/TACO -type f \( -name "*.jpg" -o -name "*.JPG" -o -name "*.png" -o -name "*.PNG" \) | head -1)
echo "Demo image: $IMG"

python scripts/infer/infer_image.py \
  --data-config "$DATA_CONFIG" \
  --model-config configs/models/hybrid_yolov8n_effb0.yaml \
  --detector-weights "$DET" \
  --classifier-weights "$CLF" \
  --image "$IMG" \
  --save-dir outputs/predictions/demo_taco \
  --use-tta \
  --use-wbf

# Optional external images:
if [ -d /kaggle/input/new-waste-images ]; then
  for IMG in /kaggle/input/new-waste-images/*; do
    python scripts/infer/infer_image.py \
      --data-config "$DATA_CONFIG" \
      --model-config configs/models/hybrid_yolov8n_effb0.yaml \
      --detector-weights "$DET" \
      --classifier-weights "$CLF" \
      --image "$IMG" \
      --save-dir outputs/predictions/new_images \
      --use-tta \
      --use-wbf || true
  done
fi


## 8. Grad-CAM cho classifier wrong predictions


In [ ]:
from pathlib import Path
import os

RUN_GRADCAM = False  # đổi True sau khi evaluate_classifier đã tạo wrong_predictions.json
RUN_NAME = "run4_balanced_sampler"

if not RUN_GRADCAM:
    print("RUN_GRADCAM=False → skip Grad-CAM.")
else:
    import cv2
    import torch
    from PIL import Image

    from waste_detection.models.efficientnet_classifier import EfficientNetB0Classifier
    from waste_detection.data.transforms import ClassifierTransformFactory
    from waste_detection.evaluation.grad_cam import GradCAMExplainer
    from waste_detection.utils.device import get_device
    from waste_detection.utils.io import IOUtils

    weights_path = Path(f"outputs/checkpoints/efficientnet_b0/{RUN_NAME}/best.pth")
    wrong_path = Path(f"outputs/metrics/classifier/{RUN_NAME}/test/wrong_predictions.json")
    save_dir = Path(f"outputs/figures/gradcam/{RUN_NAME}")
    IOUtils.ensure_dir(save_dir)

    if not weights_path.exists():
        raise FileNotFoundError(weights_path)
    if not wrong_path.exists():
        raise FileNotFoundError(wrong_path)

    device = get_device()
    model, class_names = EfficientNetB0Classifier.load_from_checkpoint(weights_path, map_location=device)
    model = model.to(device)
    model.eval()

    target_layer = GradCAMExplainer.get_efficientnet_last_conv_layer(model)
    explainer = GradCAMExplainer(model=model, target_layer=target_layer)
    transform = ClassifierTransformFactory.build_eval_transform(image_size=224)

    wrong_predictions = IOUtils.load_json(wrong_path)

    for idx, row in enumerate(wrong_predictions[:12]):
        image_path = Path(row["image_path"])
        image_bgr = IOUtils.load_image_bgr(image_path)
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(image_rgb)
        input_tensor = transform(pil_image).unsqueeze(0).to(device)

        save_path = save_dir / f"{idx:02d}_true_{row['true_class']}_pred_{row['pred_class']}.png"
        explainer.explain(
            input_tensor=input_tensor,
            original_image_rgb=image_rgb,
            target_class=int(row["pred_id"]),
            save_path=save_path,
        )

    print("Saved Grad-CAM:", save_dir)


## 9. Tổng hợp classifier + hybrid metrics và vẽ biểu đồ


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

Path("outputs/metrics/final").mkdir(parents=True, exist_ok=True)
Path("outputs/figures/final").mkdir(parents=True, exist_ok=True)

classifier_runs = ["run1_baseline", "run2_strong_aug", "run3_freeze_cosine_warmup", "run4_balanced_sampler"]
rows = []
for run in classifier_runs:
    p = Path(f"outputs/metrics/classifier/{run}/test/metrics.json")
    if not p.exists():
        print("Missing:", p)
        continue
    m = json.loads(p.read_text(encoding="utf-8"))
    rows.append({
        "run": run,
        "accuracy": m.get("accuracy"),
        "macro_f1": m.get("macro_f1"),
        "weighted_f1": m.get("weighted_f1"),
        "macro_precision": m.get("macro_precision"),
        "macro_recall": m.get("macro_recall"),
    })

df_cls = pd.DataFrame(rows)
df_cls.to_csv("outputs/metrics/final/classifier_summary.csv", index=False)
display(df_cls)

if not df_cls.empty:
    plt.figure(figsize=(10, 5))
    plt.bar(df_cls["run"], df_cls["macro_f1"])
    plt.xticks(rotation=25, ha="right")
    plt.ylabel("Macro F1")
    plt.title("EfficientNet-B0 classifier Macro F1 by run")
    plt.tight_layout()
    plt.savefig("outputs/figures/final/classifier_macro_f1_comparison.png", dpi=300)
    plt.show()

hybrid_runs = ["H1_run1_baseline", "H2_run2_strong_aug", "H3_run3_freeze_cosine_warmup", "H4_run4_balanced_sampler"]
rows = []
for run in hybrid_runs:
    p = Path(f"outputs/metrics/hybrid/{run}/test/metrics.json")
    if not p.exists():
        print("Missing:", p)
        continue
    m = json.loads(p.read_text(encoding="utf-8"))
    rows.append({
        "run": run,
        "precision": m.get("precision"),
        "recall": m.get("recall"),
        "f1": m.get("f1"),
        "map50": m.get("map50"),
        "map50_95": m.get("map50_95"),
        "classification_accuracy_on_matched": m.get("classification_accuracy_on_matched"),
        "average_inference_time_seconds": m.get("average_inference_time_seconds"),
        "fps": m.get("fps"),
    })

df_hyb = pd.DataFrame(rows)
df_hyb.to_csv("outputs/metrics/final/hybrid_summary.csv", index=False)
display(df_hyb)

if not df_hyb.empty:
    plt.figure(figsize=(10, 5))
    plt.bar(df_hyb["run"], df_hyb["map50"])
    plt.xticks(rotation=25, ha="right")
    plt.ylabel("Hybrid mAP50")
    plt.title("Hybrid end-to-end mAP50 by run")
    plt.tight_layout()
    plt.savefig("outputs/figures/final/hybrid_map50_comparison.png", dpi=300)
    plt.show()


## 10. Optional — upload evaluation summary lên W&B


In [ ]:
import os
from pathlib import Path

if os.environ.get("RUN_WANDB_EVAL_UPLOAD") != "1":
    print("RUN_WANDB_EVAL_UPLOAD=0 → skip.")
else:
    import wandb
    import pandas as pd

    entity = os.environ.get("WANDB_ENTITY") or None
    wandb.init(
        project="waste-detection-taco",
        entity=entity,
        name="hybrid_evaluation_summary",
        group="final_summary",
        job_type="evaluation",
    )

    for path in [
        Path("outputs/metrics/final/classifier_summary.csv"),
        Path("outputs/metrics/final/hybrid_summary.csv"),
    ]:
        if path.exists():
            wandb.log({path.stem: wandb.Table(dataframe=pd.read_csv(path))})

    for image_path in Path("outputs/figures").rglob("*.png"):
        wandb.log({str(image_path): wandb.Image(str(image_path))})

    wandb.finish()


## 11. Đóng gói outputs


In [ ]:
%%bash
set -e

cd "$REPO_DIR"

mkdir -p /kaggle/working/artifacts

tar -czf /kaggle/working/artifacts/hybrid_metrics_figures_predictions.tar.gz \
  outputs/metrics \
  outputs/figures \
  outputs/predictions \
  outputs/logs \
  2>/dev/null || true

tar -czf /kaggle/working/artifacts/hybrid_checkpoints.tar.gz \
  outputs/checkpoints/hybrid_yolov8n_effb0 \
  outputs/checkpoints/efficientnet_b0 \
  2>/dev/null || true

ls -lh /kaggle/working/artifacts
